# Baseline sensitivity: data-quality ablations

Re-runs the logistic-regression baseline (same pipeline as `log-reg-model.ipynb`, train 2024 / test 2025)
with and without three groups of questionable rows:

1. games missing attendance (14% of the cleaned dataset)
2. over-capacity games (`attendance > capacity`, 18% of modelled games)
3. games with imputed FCS-opponent Elo (`awayEloMissing`, 14% of modelled games)

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

clean = pd.read_csv("../data/full_dataset_clean.csv")
feats = pd.read_csv("../data/features.csv")
flags = ["flag_over_capacity", "awayEloMissing", "homeEloMissing"]
df = feats.merge(clean[["id"] + flags], on="id", how="left", validate="1:1")
assert len(df) == len(feats)
print(df.shape)

(1673, 43)


## 1. Missing attendance

These games have no `high_attendance` label, so they cannot be scored. Check they are already out of the baseline.

In [2]:
print(f"cleaned dataset: {clean['flag_missing_attendance'].mean():.1%} of games missing attendance "
      f"({clean['flag_missing_attendance'].sum()} of {len(clean)}); {clean['high_attendance'].isna().sum()} unlabelable in total")
print("missing-attendance games in features.csv:", clean[clean['id'].isin(feats['id'])]['flag_missing_attendance'].sum())
assert not clean[clean["id"].isin(feats["id"])]["flag_missing_attendance"].any()

cleaned dataset: 14.2% of games missing attendance (286 of 2021); 348 unlabelable in total
missing-attendance games in features.csv: 0


**Result:** 0 rows. The baseline already excludes missing-attendance games, so "with" is not evaluable
without imputing a target, and "without" is the baseline itself. No re-run needed.

## 2. Baseline vs. ablations

In [3]:
TARGET = "high_attendance"
X = df.drop(columns=["id", "season", TARGET] + flags)
y = df[TARGET]
cat_cols = X.select_dtypes(include=["object", "str", "bool", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# same settings as the baseline in log-reg-model.ipynb
def make_pipeline():
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def run(name, train_keep, test_keep):
    tr = (df["season"] == 2024) & train_keep
    te = (df["season"] == 2025) & test_keep
    p = make_pipeline().fit(X[tr], y[tr])
    pred, proba = p.predict(X[te]), p.predict_proba(X[te])[:, 1]
    return {"scenario": name, "n_train": tr.sum(), "n_test": te.sum(),
            "train_high_rate": y[tr].mean(), "test_high_rate": y[te].mean(),
            "accuracy": accuracy_score(y[te], pred), "f1": f1_score(y[te], pred),
            "roc_auc": roc_auc_score(y[te], proba)}

all_rows = pd.Series(True, index=df.index)
not_over = ~df["flag_over_capacity"]
not_fcs_elo = ~(df["awayEloMissing"] | df["homeEloMissing"])

results = pd.DataFrame([
    run("baseline (all rows)", all_rows, all_rows),
    run("drop over-capacity (train+test)", not_over, not_over),
    run("drop over-capacity (train only)", not_over, all_rows),
    run("drop imputed-Elo (train+test)", not_fcs_elo, not_fcs_elo),
    run("drop imputed-Elo (train only)", not_fcs_elo, all_rows),
    run("drop both (train+test)", not_over & not_fcs_elo, not_over & not_fcs_elo),
]).set_index("scenario")
base = results.loc["baseline (all rows)"]
for m in ["accuracy", "f1", "roc_auc"]:
    results[f"d_{m}"] = results[m] - base[m]
results.round(3)

,n_train,n_test,train_high_rate,test_high_rate,accuracy,f1,roc_auc,d_accuracy,d_f1,d_roc_auc
scenario,,,,,,,,,,
baseline (all rows),815,858,0.450,0.437,0.815,0.789,0.893,0.000,0.000,0.000
drop over-capacity (train+test),658,716,0.319,0.325,0.788,0.648,0.860,-0.027,-0.141,-0.033
drop over-capacity (train only),658,858,0.319,0.437,0.790,0.738,0.878,-0.024,-0.050,-0.015
drop imputed-Elo (train+test),704,732,0.464,0.445,0.818,0.796,0.893,0.004,0.007,-0.000
drop imputed-Elo (train only),704,858,0.464,0.437,0.818,0.794,0.892,0.003,0.005,-0.001
drop both (train+test),560,609,0.327,0.333,0.788,0.660,0.859,-0.027,-0.129,-0.034


## Recorded results (test = 2025, logistic regression baseline)

| Scenario | n train | n test | Test high rate | Accuracy | F1 | ROC-AUC | Δ Acc | Δ F1 | Δ AUC |
|---|---|---|---|---|---|---|---|---|---|
| Baseline (all rows) | 815 | 858 | 0.437 | 0.815 | 0.789 | 0.893 | - | - | - |
| Drop over-capacity (train+test) | 658 | 716 | 0.325 | 0.788 | 0.648 | 0.860 | -0.027 | -0.141 | -0.033 |
| Drop over-capacity (train only) | 658 | 858 | 0.437 | 0.790 | 0.738 | 0.878 | -0.024 | -0.050 | -0.015 |
| Drop imputed Elo (train+test) | 704 | 732 | 0.445 | 0.818 | 0.796 | 0.893 | +0.004 | +0.007 | 0.000 |
| Drop imputed Elo (train only) | 704 | 858 | 0.437 | 0.818 | 0.794 | 0.892 | +0.003 | +0.005 | -0.001 |
| Drop both (train+test) | 560 | 609 | 0.333 | 0.788 | 0.660 | 0.859 | -0.027 | -0.129 | -0.034 |
| Missing attendance | n/a | n/a | n/a | n/a | n/a | n/a | 0 rows in baseline, no effect | | |

## Findings

- **Missing attendance:** no effect. Those games are unlabeled and already absent from the baseline (0 of 1,673 rows).
- **Over-capacity (18% of modelled games):** dropping them lowers every metric (ROC-AUC -0.015 to -0.033, F1 -0.05 to -0.14). They are all  by construction (any cutoff <= 1.0), so removing them cuts the positive rate from ~44% to ~32% and removes the easiest positives. The baseline is partly propping up its score with these games; keep them, since they are valid high-attendance examples, but treat the harder 'drop' numbers as the more conservative estimate.
- **Imputed FCS-opponent Elo (14%):** negligible effect (accuracy +0.003 to +0.004, F1 +0.005 to +0.007, ROC-AUC about 0). The imputation is not driving results.

Note the train+test variants change the test set, so their metrics are not strictly comparable with the baseline; the train-only variants score on the same 2025 test rows.